# 02 · Silver Pipeline

**Status: validated Silver checkpoint; Gold modelling is next.**

This notebook explains the decisions and inspects their evidence. [The pipeline](../src/daltix_case/pipelines/silver_pipeline.py) owns execution; [the Silver modules](../src/daltix_case/silver/) own every transformation.

Read each table's **grain → source issues → treatment → contract**, inspect the results, then skim its conclusion. The [README](../README.md) holds the longer rationale; [01 · Discovery](01_source_discovery.ipynb) retains the exploratory history.

Raw stays immutable. Missing attributes remain nullable, primary values win, and uncertainty stays visible. Natural keys remain in Silver; dimensional modelling belongs to Gold.


## 1. Load a Validated Checkpoint

Keep `RUN_PIPELINE = False` to inspect existing outputs. Set it to `True` to rebuild all seven tables through `run_silver_pipeline()`; the same entry point is available from the CLI. Missing, changed or incomplete artifacts fail explicitly.

No database access is required. The optional environment paths support isolated validation; normal runs use the project's `data/raw` and `data/clean`.


In [1]:
# Locate the project and import the official pipeline, keeping transformations in src.
import os
from pathlib import Path

import duckdb
import polars as pl
from IPython.display import display

from daltix_case.pipelines.silver_pipeline import (
    OUTPUTS,
    load_silver_manifest,
    run_silver_pipeline,
)
from daltix_case.silver import SILVER_FILES
from daltix_case.source_io import validate_local_snapshot

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file() and (path / "src/daltix_case").is_dir()
)
RAW_DIR = Path(os.environ.get("DALTIX_RAW_DIR", PROJECT_ROOT / "data/raw"))
SILVER_DIR = Path(os.environ.get("DALTIX_SILVER_DIR", PROJECT_ROOT / "data/clean"))
RUN_PIPELINE = False
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(8)

polars.config.Config

In [2]:
# Build only on explicit request, then verify code, Raw lineage and all published files.
if RUN_PIPELINE:
    run_silver_pipeline(RAW_DIR, SILVER_DIR)
manifest = load_silver_manifest(SILVER_DIR)
raw_manifest = validate_local_snapshot(RAW_DIR)
if manifest["raw_files"] != raw_manifest["files"]:
    raise RuntimeError(
        "This Silver checkpoint does not correspond to the current Raw snapshot."
    )
metrics = manifest["metrics"]
pl.DataFrame(
    [
        {"dataset": f["dataset"], "rows": f["rows"], "file": f["file"]}
        for f in manifest["files"]
    ]
)

dataset,rows,file
str,i64,str
"""products""",32826,"""silver_products.parquet"""
"""locations""",1638,"""silver_locations.parquet"""
"""prices""",1198547,"""silver_prices.parquet"""
"""nutritionals""",9467978,"""silver_nutritionals.parquet"""
"""weekly_products""",114517,"""silver_weekly_products.parquet"""
"""weekly_locations""",1230,"""silver_weekly_locations.parque…"
"""weekly_prices""",19075099,"""silver_weekly_prices.parquet"""


### 1.1 Reading the Evidence

Hard contracts stop publication on structural failure. DQ flags describe retained exceptions. Coverage and missingness are measured separately from validity. Every metric below comes from the published checkpoint or a read-only query; no second transformation implementation lives here.


In [3]:
# Register lazy Parquet views and small display helpers without collecting the large facts.
duck = duckdb.connect()
source_names = {
    "weekly_products": "weekly_prices_products",
    "weekly_locations": "weekly_prices_locations",
}
for name in OUTPUTS:
    duck.read_parquet(str(SILVER_DIR / SILVER_FILES[name])).create_view(name)
    source = source_names.get(name, name)
    duck.read_parquet(str(RAW_DIR / f"{source}.parquet")).create_view(f"raw_{name}")


def show_metrics(name, keys=None):
    values = metrics[name]
    return pl.DataFrame(
        {
            "metric": list(keys or values),
            "value": [str(values[key]) for key in (keys or values)],
        }
    )


def missingness(raw, silver, columns):
    return pl.DataFrame(
        {
            "column": columns,
            "raw_nulls": [raw[c].null_count() for c in columns],
            "silver_nulls": [silver[c].null_count() for c in columns],
        }
    )


def converted_tokens(raw, silver, columns):
    frames = []
    for column in columns:
        converted = (
            raw.select("daltix_id", pl.col(column).alias("raw_value"))
            .join(
                silver.select("daltix_id", pl.col(column).alias("silver_value")),
                on="daltix_id",
                validate="1:1",
            )
            .filter(
                pl.col("raw_value").is_not_null() & pl.col("silver_value").is_null()
            )
        )
        frames.append(
            converted.group_by("raw_value")
            .len()
            .with_columns(pl.lit(column).alias("column"))
        )
    return (
        pl.concat(frames)
        .select("column", "raw_value", "len")
        .sort("column", "raw_value")
    )


def checkpoint_result(name):
    record = next(f for f in manifest["files"] if f["dataset"] == name)
    return {
        "dataset": name,
        "persisted_rows": record["rows"],
        "contracts": "passed before publication",
    }

## 2. Products

**Purpose / grain:** a nullable product reference; one row per `daltix_id` within this source.

**Issues → treatment:** normalize confirmed placeholders, retain country/language, and preserve descriptive gaps. `NAN` is a real brand, not a missing token; `#N/A` is missing.

**Contract:** required source schema, preserved row count, complete source context and unique ID. [Implementation](../src/daltix_case/silver/products.py).


In [4]:
# Inspect the source reference before comparing it with the published result.
products_raw = pl.read_parquet(RAW_DIR / "products.parquet")
products_raw.head()

daltix_id,shop,name,brand,country,description,categories,language
str,str,str,str,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""idla""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""","""lu""","""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr"""
"""c38129dc388664ce48e060eebb9598…","""frc""","""Axe BodySpray Deodorant You 15…","""axe""","""be""",null,"""[ [ ""Verzorging & hygiën…","""nl"""
"""15fbf0a66ff12e20fde41913aebd72…","""idla""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""","""be""","""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl"""
"""7c93bb76b2772b001b264b779cb60c…","""ha""","""Blistex Lip relief crème""","""blistex""","""nl""","""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl"""
"""d60208deb02f76de96a8f771b1652d…","""frc""","""Cartec Achteruitkijkspiegel me…","""cartec""","""be""","""""","""[ [ ""Niet-voeding"" ], …","""nl"""


In [5]:
# Read the official product output, keeping the notebook free of cleaning rules.
products = pl.read_parquet(SILVER_DIR / SILVER_FILES["products"])
products.head()

daltix_id,shop,name,brand,country,description,categories,language
str,str,str,str,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""idla""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""","""lu""","""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr"""
"""c38129dc388664ce48e060eebb9598…","""frc""","""Axe BodySpray Deodorant You 15…","""axe""","""be""",null,"""[ [ ""Verzorging & hygiën…","""nl"""
"""15fbf0a66ff12e20fde41913aebd72…","""idla""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""","""be""","""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl"""
"""7c93bb76b2772b001b264b779cb60c…","""ha""","""Blistex Lip relief crème""","""blistex""","""nl""","""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl"""
"""d60208deb02f76de96a8f771b1652d…","""frc""","""Cartec Achteruitkijkspiegel me…","""cartec""","""be""",null,"""[ [ ""Niet-voeding"" ], …","""nl"""


In [6]:
# Separate literal Raw nulls from the assessed missingness represented in Silver.
attributes = ["name", "brand", "description", "categories"]
missingness(products_raw, products, attributes)

column,raw_nulls,silver_nulls
str,i64,i64
"""name""",41,42
"""brand""",740,1279
"""description""",3393,8013
"""categories""",497,497


In [7]:
# List the actual source tokens converted to null so information loss is visible.
product_tokens = converted_tokens(products_raw, products, attributes)
product_tokens

column,raw_value,len
str,str,u32
"""brand""","""""",539
"""description""","""""",4620
"""name""","""#N/A""",1


In [8]:
# Retain the description-token check and confirm that legitimate NAN brands survive.
display(product_tokens.filter(pl.col("column") == "description"))
products.filter(pl.col("brand").str.to_lowercase() == "nan").select(
    "daltix_id", "name", "brand"
)

column,raw_value,len
str,str,u32
"""description""","""""",4620


daltix_id,name,brand
str,str,str
"""56fa6176fccc2cf68ca95d6e6ae09e…","""NAN Optipro Groeimelk 1+ vanaf…","""nan"""
"""febed5d0fd1968156b23d4ae5d3af2…","""NAN Optipro Groeimelk 1+ vanaf…","""nan"""


In [9]:
# Show key integrity and the persisted contract checkpoint alongside the missingness metrics.
display(
    products.select(
        pl.len().alias("rows"), pl.col("daltix_id").n_unique().alias("business_keys")
    )
)
display(show_metrics("products"))
checkpoint_result("products")

rows,business_keys
u32,u32
32826,32826


metric,value
str,str
"""rows""","""32826"""
"""missing_name""","""42"""
"""missing_brand""","""1279"""
"""missing_description""","""8013"""
"""missing_categories""","""497"""


{'dataset': 'products',
 'persisted_rows': 32826,
 'contracts': 'passed before publication'}

### 2.1 Conclusion

**Established:** 32,826 product rows; the business key remains unique.

**Caveats:** 42 names, 1,279 brands, 8,013 descriptions and 497 categories remain missing.

**Silver decision:** Keep optional attributes nullable. Use this source for compatible fallback only; do not infer universal ID or historical validity.


## 3. Locations

**Purpose / grain:** one location within a retailer; working key `(shop, id)`.

**Issues → treatment:** preserve both records in the known collision, mark them unsafe for enrichment, and omit the entirely empty `type`. Keep nullable postcode/coordinates and explicit country.

**Contract:** row preservation, complete key/context, finite in-range coordinates when present and consistent coordinate pairs. Uniqueness is **not** forced. [Implementation](../src/daltix_case/silver/locations.py).


In [10]:
# Inspect Raw fields, including the empty type column retained in the immutable source.
duck.sql("SELECT * FROM raw_locations LIMIT 5").pl()

shop,country_code,id,type,geolocation_latitude,geolocation_longitude,postcode,sources
str,str,str,str,f64,f64,str,str
"""ldil""","""nl""","""1a131""",null,null,null,null,"""[ ""online"" ]"""
"""ldil""","""de""","""5f02f""",null,null,null,null,"""[ ""online"" ]"""
"""gc""","""be""","""f02af""",null,51.015406,3.6528982,"""9051""","""[ ""online"" ]"""
"""frc""","""be""","""64003""",null,50.908945,4.3048594,"""1780""","""[ ""online"" ]"""
"""frc""","""be""","""dd6cf""",null,50.492171,3.9675296,"""7010""","""[ ""online"" ]"""


In [11]:
# Inspect the cleaned reference and its retained business keys.
locations = pl.read_parquet(SILVER_DIR / SILVER_FILES["locations"])
locations.head()

shop,country_code,id,geolocation_latitude,…,is_enrichment_safe,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,str,f64,…,bool,bool,bool,bool
"""ldil""","""nl""","""1a131""",null,…,true,false,false,false
"""ldil""","""de""","""5f02f""",null,…,true,false,false,false
"""gc""","""be""","""f02af""",51.015406,…,true,false,false,false
"""frc""","""be""","""64003""",50.908945,…,true,false,false,false
"""frc""","""be""","""dd6cf""",50.492171,…,true,false,false,false


In [12]:
# Keep the two conflicting location records visible rather than choosing an arbitrary winner.
locations.filter(pl.col("dq_business_key_collision"))

shop,country_code,id,geolocation_latitude,…,is_enrichment_safe,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,str,f64,…,bool,bool,bool,bool
"""lld""","""be""","""f334d""",50.601871,…,false,false,false,false
"""lld""","""be""","""f334d""",50.452781,…,false,false,false,false


In [13]:
# Inspect geographic flags on the persisted output.
locations.select(
    "shop",
    "id",
    "dq_invalid_latitude",
    "dq_invalid_longitude",
    "dq_incomplete_coordinates",
).head()

shop,id,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,bool,bool,bool
"""ldil""","""1a131""",false,false,false
"""ldil""","""5f02f""",false,false,false
"""gc""","""f02af""",false,false,false
"""frc""","""64003""",false,false,false
"""frc""","""dd6cf""",false,false,false


In [14]:
# Measure retained gaps and geographic validity without treating missing as invalid.
display(show_metrics("locations"))
locations.select(
    pl.col("dq_incomplete_coordinates").sum().alias("incomplete_coordinate_rows")
)

metric,value
str,str
"""rows""","""1638"""
"""business_key_collision_rows""","""2"""
"""missing_postcode""","""40"""
"""missing_coordinates""","""18"""
"""invalid_latitudes""","""0"""
"""invalid_longitudes""","""0"""


incomplete_coordinate_rows
u32
0


In [15]:
# Show why the source key is a working key rather than a global uniqueness contract.
display(
    locations.select(
        pl.len().alias("rows"),
        pl.struct("shop", "id").n_unique().alias("business_keys"),
    )
)
checkpoint_result("locations")

rows,business_keys
u32,u32
1638,1637


{'dataset': 'locations',
 'persisted_rows': 1638,
 'contracts': 'passed before publication'}

### 3.1 Conclusion

**Established:** 1,638 rows and 1,637 shop/ID keys; both `lld + f334d` records are retained.

**Caveats:** 40 missing postcodes and 18 missing coordinate pairs; no out-of-range coordinates.

**Silver decision:** Exclude ambiguous keys from automatic fallback. Keep country explicit and preserve nullable geography.


## 4. Prices

**Purpose / grain:** a collected price at `(daltix_id, shop, location, downloaded_on)`.

**Issues → treatment:** retain meaningful null promotional prices. The adopted source convention interprets null as no active promotion; present values below regular price set `is_promotion`. This convention is source-specific.

**Contract:** exact key uniqueness, complete required fields, finite positive prices and preserved rows. Invalid numeric text must fail, not silently become null. [Implementation](../src/daltix_case/silver/prices.py).


In [16]:
# Inspect Raw prices without loading the complete fact into Python.
duck.sql("SELECT * FROM raw_prices LIMIT 5").pl()

daltix_id,shop,country,location,…,promo_price,unit_std,currency,downloaded_on
str,str,str,str,…,f64,str,str,date
"""40f23e8eddf2cd9f28b1c50e5d856c…","""frc""","""be""","""f2177""",…,null,"""su""","""eur""",2020-04-17
"""928fe14059bb99a4ed1613b39f887f…","""frc""","""be""","""9ae3b""",…,null,"""su""","""eur""",2020-04-17
"""1f004043ca1559c432904afa20db08…","""frc""","""be""","""dd6cf""",…,null,"""su""","""eur""",2020-04-17
"""77eb025c8484adb9bda90be7f8c657…","""frc""","""be""","""7f6c1""",…,null,"""su""","""eur""",2020-04-17
"""28c1a2ccaa516fb33c438b23795631…","""frc""","""be""","""77446""",…,null,"""su""","""eur""",2020-04-17


In [17]:
# Inspect the official typed Silver output.
duck.sql("SELECT * FROM prices LIMIT 5").pl()

daltix_id,shop,country,location,…,is_promotion,dq_non_positive_price,dq_non_positive_promo_price,dq_promo_not_below_price
str,str,str,str,…,bool,bool,bool,bool
"""40f23e8eddf2cd9f28b1c50e5d856c…","""frc""","""be""","""f2177""",…,false,false,false,false
"""928fe14059bb99a4ed1613b39f887f…","""frc""","""be""","""9ae3b""",…,false,false,false,false
"""1f004043ca1559c432904afa20db08…","""frc""","""be""","""dd6cf""",…,false,false,false,false
"""77eb025c8484adb9bda90be7f8c657…","""frc""","""be""","""7f6c1""",…,false,false,false,false
"""28c1a2ccaa516fb33c438b23795631…","""frc""","""be""","""77446""",…,false,false,false,false


In [18]:
# Inspect promotion and price-quality flags independently of the transformation.
duck.sql(
    "SELECT price, promo_price, is_promotion, dq_non_positive_price, dq_non_positive_promo_price, dq_promo_not_below_price FROM prices LIMIT 5"
).pl()

price,promo_price,is_promotion,dq_non_positive_price,dq_non_positive_promo_price,dq_promo_not_below_price
f64,f64,bool,bool,bool,bool
3.025,null,false,false,false,false
2.915,null,false,false,false,false
1.298,null,false,false,false,false
2.145,null,false,false,false,false
2.959,null,false,false,false,false


In [19]:
# Retain the exact grain check as visible evidence.
duck.sql(
    "SELECT count(*) AS rows, count(DISTINCT (daltix_id, shop, location, downloaded_on)) AS business_keys FROM prices"
).pl()

rows,business_keys
i64,i64
1198547,1198547


In [20]:
# Report promotion metrics, units and date coverage from the persisted data.
display(show_metrics("prices"))
duck.sql("""
    SELECT count(DISTINCT unit_std) AS distinct_units, count(DISTINCT currency) AS distinct_currencies,
        min(downloaded_on) AS first_date, max(downloaded_on) AS last_date,
        count(*) FILTER (WHERE dq_non_positive_price) AS non_positive_prices,
        count(*) FILTER (WHERE dq_non_positive_promo_price) AS non_positive_promos
    FROM prices
""").pl()

metric,value
str,str
"""rows""","""1198547"""
"""rows_without_promo_price""","""1191783"""
"""promotion_rows""","""6764"""
"""unexpected_promo_rows""","""0"""
"""distinct_products""","""116"""
"""distinct_locations""","""136"""


distinct_units,distinct_currencies,first_date,last_date,non_positive_prices,non_positive_promos
i64,i64,date,date,i64,i64
1,1,2020-02-25,2021-02-25,0,0


In [21]:
# Confirm that the displayed data belongs to a completed publication.
checkpoint_result("prices")

{'dataset': 'prices',
 'persisted_rows': 1198547,
 'contracts': 'passed before publication'}

### 4.1 Conclusion

**Established:** 1,198,547 rows; 6,764 promotions and no unexpected populated promotional prices.

**Caveats:** 1,191,783 promotional prices remain null; the source convention is an adopted interpretation, not proof from null frequency alone.

**Silver decision:** Keep nulls, explicit unit/currency and the full observation key. Do not harmonize with weekly promotion semantics by simple coalescing.


## 5. Nutritionals

**Purpose / grain:** long nutrient observations at `(daltix_id, shop, country, download_date, nutrient_name)`.

**Issues → treatment:** remove only exact duplicates, then select one source version per product/shop/country/date using the existing completeness-first ranking. Preserve conflict flags, selected payload hash and portion basis; Raw keeps all versions.

**Contract:** valid nonempty nutrient objects, usable portion context, complete source keys, unique long grain and row/observation reconciliation. Unexpected nutrient units are flagged, never converted by guessing. [Implementation](../src/daltix_case/silver/nutritionals.py).


In [22]:
# Inspect the original nested source and its schema before reading the normalized output.
display(duck.sql("DESCRIBE raw_nutritionals").pl())
duck.sql("SELECT * FROM raw_nutritionals LIMIT 5").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""daltix_id""","""VARCHAR""","""YES""",null,null,null
"""shop""","""VARCHAR""","""YES""",null,null,null
"""country""","""VARCHAR""","""YES""",null,null,null
"""download_date""","""DATE""","""YES""",null,null,null
"""nutritional_values_std""","""VARCHAR""","""YES""",null,null,null
"""language""","""VARCHAR""","""YES""",null,null,null


daltix_id,shop,country,download_date,nutritional_values_std,language
str,str,str,date,str,str
"""8d93eace1a61d0644680915e05ce88…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""b13ea65799dc89cf764c8759082d25…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""447fa714c27770edae98dcf63c31c5…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""327ea94401fa7b2eee762b710f08bc…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""a4a10f28bc35e883d6caafff75631a…","""lld""","""be""",2021-01-27,"""{ ""nutrients"": { ""carboh…","""nl"""


In [23]:
# Inspect the selected source context without recomputing canonicalization.
duck.sql(
    "SELECT DISTINCT daltix_id, shop, country, language, download_date FROM nutritionals LIMIT 5"
).pl()

daltix_id,shop,country,language,download_date
str,str,str,str,date
"""6aceee84ba1bfd86a581aabfc1d21a…","""ha""","""nl""","""nl""",2021-01-01
"""6b07422be060a98b45a13550b08e89…","""ha""","""nl""","""nl""",2021-01-02
"""6b37f8954ba9fe1eb75cc1e488215b…","""lld""","""be""","""nl""",2021-01-30
"""6b626367a6a5133bc2e209b9e9c658…","""ha""","""nl""","""nl""",2021-01-05
"""6b82ae447fbf9cc614b2fe7e912ddd…","""frc""","""be""","""nl""",2021-02-20


In [24]:
# Retain the Raw JSON validity and structure evidence alongside the stronger pipeline contract.
duck.sql("""
    SELECT count(*) AS rows,
        count(*) FILTER (WHERE nutritional_values_std IS NULL) AS missing_json_rows,
        count(*) FILTER (WHERE NOT json_valid(nutritional_values_std)) AS invalid_json_rows,
        count(*) FILTER (WHERE json_type(nutritional_values_std, '$.nutrients') IS DISTINCT FROM 'OBJECT') AS invalid_nutrient_objects
    FROM raw_nutritionals
""").pl()

rows,missing_json_rows,invalid_json_rows,invalid_nutrient_objects
i64,i64,i64,i64
1096542,0,0,0


In [25]:
# Report source deduplication and canonical observation counts from the official run.
show_metrics(
    "nutritionals",
    [
        "exact_duplicates_removed",
        "canonical_observations",
        "resolved_repeated_grains",
        "resolved_nutritional_conflicts",
        "resolved_language_conflicts",
        "hash_tiebreak_grains",
    ],
)

metric,value
str,str
"""exact_duplicates_removed""","""0"""
"""canonical_observations""","""1096442"""
"""resolved_repeated_grains""","""100"""
"""resolved_nutritional_conflicts""","""100"""
"""resolved_language_conflicts""","""0"""
"""hash_tiebreak_grains""","""91"""


In [26]:
# Inspect the selected versions and the provenance explaining each choice.
duck.sql("""
    SELECT DISTINCT daltix_id, shop, country, download_date, source_payload_hash,
        numeric_nutrient_count, nutrient_count, populated_unit_count, nutrition_selection_reason
    FROM nutritionals WHERE dq_repeated_grain
    ORDER BY daltix_id, shop, country, download_date LIMIT 10
""").pl()

daltix_id,shop,country,download_date,…,numeric_nutrient_count,nutrient_count,populated_unit_count,nutrition_selection_reason
str,str,str,date,…,i64,i64,i64,str
"""05766389442155ce47fa08689d547c…","""ha""","""nl""",2020-12-18,…,9,9,9,"""deterministic_hash_tiebreak"""
"""08e74e5c36f50fde38757429acf3bb…","""frc""","""be""",2021-02-02,…,9,9,9,"""completeness_first"""
"""17c65f953f988a273400c335cd7e73…","""ha""","""nl""",2020-12-29,…,9,9,9,"""deterministic_hash_tiebreak"""
"""17d33e7b62170c7f2d957e0b365316…","""frc""","""be""",2021-02-04,…,9,9,9,"""deterministic_hash_tiebreak"""
"""182ab99e8d1c6f56126dda5d9c3d8e…","""ha""","""nl""",2020-11-27,…,9,9,9,"""deterministic_hash_tiebreak"""
"""18a94ff31934e33d3df52eae9cd195…","""frc""","""be""",2021-02-08,…,9,9,9,"""deterministic_hash_tiebreak"""
"""199328fdfdf7b3385c951fc265aa48…","""frc""","""be""",2021-02-05,…,8,8,8,"""deterministic_hash_tiebreak"""
"""1a112b3283f106e1f7ea6c77307bed…","""ha""","""nl""",2020-12-04,…,9,9,9,"""deterministic_hash_tiebreak"""
"""1b342df565cbb5dee3600c3e54eb7c…","""ha""","""be""",2020-12-29,…,9,9,9,"""deterministic_hash_tiebreak"""


In [27]:
# Separate genuine completeness wins from deterministic ties instead of calling every choice resolved.
duck.sql("""
    SELECT nutrition_selection_reason,
        count(DISTINCT (daltix_id, shop, country, download_date)) AS observations
    FROM nutritionals GROUP BY nutrition_selection_reason ORDER BY observations DESC
""").pl()

nutrition_selection_reason,observations
str,i64
"""single_observation""",1096342
"""deterministic_hash_tiebreak""",91
"""completeness_first""",9


In [28]:
# Inspect the long representation together with the portion required to interpret its values.
duck.sql(
    "SELECT daltix_id, download_date, nutrient_name, nutrient_value_raw, nutrient_value, nutrient_unit, portion_value, portion_unit FROM nutritionals LIMIT 15"
).pl()

daltix_id,download_date,nutrient_name,nutrient_value_raw,nutrient_value,nutrient_unit,portion_value,portion_unit
str,date,str,str,f64,str,f64,str
"""3fe91759e53178fcd7c3a0cf49cbba…",2020-12-30,"""sugars""","""3.5""",3.5,"""g""",100.0,"""g"""
"""3fe91759e53178fcd7c3a0cf49cbba…",2021-01-04,"""sugars""","""3.5""",3.5,"""g""",100.0,"""g"""
"""3fe939b0bf761d0e803fe7184562ea…",2021-02-10,"""sugars""","""0""",0.0,"""g""",100.0,"""g"""
"""3feaa6b285d21b9757c2832bfa3421…",2020-12-09,"""sugars""","""3.1""",3.1,"""g""",100.0,"""g"""
"""3feaa6b285d21b9757c2832bfa3421…",2020-12-20,"""sugars""","""3.1""",3.1,"""g""",100.0,"""g"""
"""3feaa6b285d21b9757c2832bfa3421…",2020-12-26,"""sugars""","""3.1""",3.1,"""g""",100.0,"""g"""
…,…,…,…,…,…,…,…
"""3feb2879b1c2e7902fd2c2f280c7b1…",2020-12-25,"""sugars""","""1""",1.0,"""g""",100.0,"""g"""
"""3fedaa90dea32c531ca745b1deb7e7…",2021-02-19,"""sugars""","""3.4""",3.4,"""g""",100.0,"""g"""


In [29]:
# Retain an exact check of the published long grain.
duck.sql(
    "SELECT count(*) AS rows, count(DISTINCT (daltix_id, shop, country, download_date, nutrient_name)) AS nutrient_grains FROM nutritionals"
).pl()

rows,nutrient_grains
i64,i64
9467978,9467978


In [30]:
# Preserve value, unit and conflict monitoring without loading millions of rows into Python.
display(show_metrics("nutritionals"))
duck.sql("""
    SELECT nutrient_name, nutrient_unit, count(*) AS rows
    FROM nutritionals WHERE dq_unexpected_nutrient_unit
    GROUP BY nutrient_name, nutrient_unit ORDER BY rows DESC, nutrient_name
""").pl()

metric,value
str,str
"""rows""","""9467978"""
"""distinct_products""","""54155"""
"""distinct_nutrients""","""14"""
"""missing_numeric_values""","""0"""
"""non_numeric_values""","""0"""
"""missing_units""","""0"""
…,…
"""exact_duplicates_removed""","""0"""
"""canonical_observations""","""1096442"""


nutrient_name,nutrient_unit,rows
str,str,i64
"""energy""","""g""",30
"""fats""","""kJ""",27
"""sugars""","""ml""",8
"""salt""","""ml""",3
"""polyunsaturated_fats""","""kJ""",1
"""unsaturated_fats""","""kJ""",1


In [31]:
# Confirm that each selected observation retains its original portion basis.
display(
    duck.sql(
        "SELECT portion_value, portion_unit, count(DISTINCT (daltix_id, shop, country, download_date)) AS observations FROM nutritionals GROUP BY portion_value, portion_unit"
    ).pl()
)
checkpoint_result("nutritionals")

portion_value,portion_unit,observations
f64,str,i64
100.0,"""g""",1034855
100.0,"""ml""",61587


{'dataset': 'nutritionals',
 'persisted_rows': 9467978,
 'contracts': 'passed before publication'}

### 5.1 Conclusion

**Established:** 9,467,978 nutrient rows, 54,155 products and 14 nutrient names; 100 conflicting source grains canonicalized.

**Caveats:** 91 conflicts need the existing hash tie-break; selection is deterministic, not proof of truth. Unexpected units remain review flags.

**Silver decision:** Keep the chosen values and conflict lineage, including 100 g / 100 ml portion context. Do not use download date as an effective-from date for historical enrichment.


## 6. Weekly Products

**Purpose / grain:** one weekly product reference per source `daltix_id`.

**Issues → treatment:** normalize missing tokens; use `products` only where shop/country/language agree. Fill missing attributes only; existing primary values always win and literal disagreements remain flagged.

**Contract:** unique IDs, preserved rows, complete context and no context-unsafe filling. Zero enrichment is a valid result. [Implementation](../src/daltix_case/silver/weekly_products.py).


In [32]:
# Inspect the primary source and the fallback used by the official run.
weekly_products_raw = pl.read_parquet(RAW_DIR / "weekly_prices_products.parquet")
display(weekly_products_raw.head())
products.head()

daltix_id,shop,name,brand,country,description,language,categories
str,str,str,str,str,str,str,str
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""lld""","""prosecco | brut | bio""","""anna perenna""","""be""","""""","""nl""","""[ [ ""Bio, Eco en Fairtra…"
"""33e67943cd5d640537699ae4303d94…","""idla""","""enrico mori® boxershorts of sl…","""enrico mori®""","""be""","""viscose/polyamide/elastaan, na…","""nl""",null
"""79995654e06c3cd0d1827d71ec95dc…","""lld""","""tandpasta | total | herstelt d…","""colgate""","""be""","""colgate tandpasta total dageli…","""nl""","""[ [ ""Hygiëne en verzorgi…"
"""7059cf765f740c27b0808e9b8bd6b2…","""idla""","""polshorloge met hartslagmeter""",null,"""be""","""draadloze overdracht van de ha…","""nl""","""[ [ ""Onze aanbiedingen"",…"
"""eb199ff50711ac7b3e98e18dca48e3…","""idla""","""biospreads""","""palazzo bio®""","""be""","""lekker als broodbeleg of om te…","""nl""",null


daltix_id,shop,name,brand,country,description,categories,language
str,str,str,str,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""idla""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""","""lu""","""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr"""
"""c38129dc388664ce48e060eebb9598…","""frc""","""Axe BodySpray Deodorant You 15…","""axe""","""be""",null,"""[ [ ""Verzorging & hygiën…","""nl"""
"""15fbf0a66ff12e20fde41913aebd72…","""idla""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""","""be""","""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl"""
"""7c93bb76b2772b001b264b779cb60c…","""ha""","""Blistex Lip relief crème""","""blistex""","""nl""","""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl"""
"""d60208deb02f76de96a8f771b1652d…","""frc""","""Cartec Achteruitkijkspiegel me…","""cartec""","""be""",null,"""[ [ ""Niet-voeding"" ], …","""nl"""


In [33]:
# Read the official weekly reference rather than implementing enrichment again.
weekly_products = pl.read_parquet(SILVER_DIR / SILVER_FILES["weekly_products"])
weekly_products.head()

daltix_id,shop,name,brand,…,is_description_enriched,dq_description_conflict,is_categories_enriched,dq_categories_conflict
str,str,str,str,…,bool,bool,bool,bool
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""lld""","""prosecco | brut | bio""","""anna perenna""",…,false,false,false,true
"""33e67943cd5d640537699ae4303d94…","""idla""","""enrico mori® boxershorts of sl…","""enrico mori®""",…,false,false,false,false
"""79995654e06c3cd0d1827d71ec95dc…","""lld""","""tandpasta | total | herstelt d…","""colgate""",…,false,false,false,false
"""7059cf765f740c27b0808e9b8bd6b2…","""idla""","""polshorloge met hartslagmeter""",null,…,false,false,false,false
"""eb199ff50711ac7b3e98e18dca48e3…","""idla""","""biospreads""","""palazzo bio®""",…,false,false,false,false


In [34]:
# Show exact primary and fallback key cardinality before interpreting enrichment coverage.
pl.DataFrame(
    [
        {
            "source": "weekly_products",
            "rows": weekly_products.height,
            "keys": weekly_products["daltix_id"].n_unique(),
        },
        {
            "source": "products",
            "rows": products.height,
            "keys": products["daltix_id"].n_unique(),
        },
    ]
)

source,rows,keys
str,i64,i64
"""weekly_products""",114517,114517
"""products""",32826,32826


In [35]:
# Retain the missingness baseline and the tokens actually normalized by the official output.
display(missingness(weekly_products_raw, weekly_products, attributes))
converted_tokens(weekly_products_raw, weekly_products, attributes)

column,raw_nulls,silver_nulls
str,i64,i64
"""name""",0,9
"""brand""",6610,9589
"""description""",0,14562
"""categories""",23294,23294


column,raw_value,len
str,str,u32
"""brand""","""""",2979
"""description""","""""",14529
"""description""","""n/a""",33
"""name""","""""",4
"""name""","""#n/a""",1
"""name""","""n/a""",4


In [36]:
# Report fallback overlap and contextual safety from the official join.
show_metrics(
    "weekly_products",
    ["fallback_matches", "safe_fallback_matches", "context_mismatches"],
)

metric,value
str,str
"""fallback_matches""","""9406"""
"""safe_fallback_matches""","""9406"""
"""context_mismatches""","""0"""


In [37]:
# Measure recovered attributes, remaining gaps and literal source disagreements.
show_metrics("weekly_products")

metric,value
str,str
"""rows""","""114517"""
"""fallback_matches""","""9406"""
"""safe_fallback_matches""","""9406"""
"""name_literal_differences""","""9356"""
"""name_case_only_differences""","""7651"""
"""brand_literal_differences""","""103"""
…,…
"""brands_enriched""","""0"""
"""descriptions_enriched""","""0"""


In [38]:
# Inspect the final reference without temporary fallback columns.
weekly_products.head()

daltix_id,shop,name,brand,…,is_description_enriched,dq_description_conflict,is_categories_enriched,dq_categories_conflict
str,str,str,str,…,bool,bool,bool,bool
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""lld""","""prosecco | brut | bio""","""anna perenna""",…,false,false,false,true
"""33e67943cd5d640537699ae4303d94…","""idla""","""enrico mori® boxershorts of sl…","""enrico mori®""",…,false,false,false,false
"""79995654e06c3cd0d1827d71ec95dc…","""lld""","""tandpasta | total | herstelt d…","""colgate""",…,false,false,false,false
"""7059cf765f740c27b0808e9b8bd6b2…","""idla""","""polshorloge met hartslagmeter""",null,…,false,false,false,false
"""eb199ff50711ac7b3e98e18dca48e3…","""idla""","""biospreads""","""palazzo bio®""",…,false,false,false,false


In [39]:
# Confirm that the published reference preserved its row count and unique ID.
checkpoint_result("weekly_products")

{'dataset': 'weekly_products',
 'persisted_rows': 114517,
 'contracts': 'passed before publication'}

### 6.1 Conclusion

**Established:** 114,517 products; 9,406 compatible fallback matches, with zero attributes filled.

**Caveats:** 9 names, 9,589 brands, 14,562 descriptions and 23,294 categories remain missing. Literal conflicts include case-only differences.

**Silver decision:** Retain primary values and safe fallback logic for future snapshots. Do not treat text disagreement alone as evidence that either source is wrong.


## 7. Weekly Locations

**Purpose / grain:** one weekly location per `(shop, location)`.

**Issues → treatment:** keep optional geography; exclude ambiguous fallback keys. Check existing postcode/coordinates for contradictions **before** filling. Fill a coordinate pair only when both primary coordinates are absent.

**Contract:** unique key, preserved rows, finite in-range complete coordinate pairs when present, and no contradictory enrichment. Names remain descriptive and nullable. [Implementation](../src/daltix_case/silver/weekly_locations.py).


In [40]:
# Inspect the primary locations and the non-weekly fallback reference.
weekly_locations_raw = pl.read_parquet(RAW_DIR / "weekly_prices_locations.parquet")
display(weekly_locations_raw.head())
locations.head()

shop,location,location_name,shop_type,…,geolocation_longitude,locality,postcode,state
str,str,str,str,…,f64,str,str,str
"""idla""","""de""","""Default""",null,…,null,null,null,null
"""idla""","""binche""","""Binche""",null,…,4.168999,"""Binche""","""7130 BINCHE""","""Wallonie"""
"""idla""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,3.3048918,"""Mouscron""","""7711""","""Wallonie"""
"""idla""","""lodelinsart""","""Lodelinsart""",null,…,4.4341463,"""Charleroi""","""6040""","""Wallonie"""
"""idla""","""péruwelz""","""Péruwelz""",null,…,3.6004418,"""Péruwelz""","""7601""","""Wallonie"""


shop,country_code,id,geolocation_latitude,…,is_enrichment_safe,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,str,f64,…,bool,bool,bool,bool
"""ldil""","""nl""","""1a131""",null,…,true,false,false,false
"""ldil""","""de""","""5f02f""",null,…,true,false,false,false
"""gc""","""be""","""f02af""",51.015406,…,true,false,false,false
"""frc""","""be""","""64003""",50.908945,…,true,false,false,false
"""frc""","""be""","""dd6cf""",50.492171,…,true,false,false,false


In [41]:
# Read the official output including context-safety flags.
weekly_locations = pl.read_parquet(SILVER_DIR / SILVER_FILES["weekly_locations"])
weekly_locations.head()

shop,location,location_name,shop_type,…,country_code,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,str,str,…,str,bool,bool,bool
"""idla""","""de""","""Default""",null,…,null,false,false,false
"""idla""","""binche""","""Binche""",null,…,null,false,false,false
"""idla""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,null,false,false,false
"""idla""","""lodelinsart""","""Lodelinsart""",null,…,null,false,false,false
"""idla""","""péruwelz""","""Péruwelz""",null,…,null,false,false,false


In [42]:
# Retain the exact retailer/location key check.
weekly_locations.select(
    pl.len().alias("rows"),
    pl.struct("shop", "location").n_unique().alias("business_keys"),
)

rows,business_keys
u32,u32
1230,1230


In [43]:
# Inspect geographic validity on the final values after any accepted enrichment.
weekly_locations.select(
    "shop",
    "location",
    "dq_invalid_latitude",
    "dq_invalid_longitude",
    "dq_incomplete_coordinates",
).head()

shop,location,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,bool,bool,bool
"""idla""","""de""",false,false,false
"""idla""","""binche""",false,false,false
"""idla""","""dottenijs2""",false,false,false
"""idla""","""lodelinsart""",false,false,false
"""idla""","""péruwelz""",false,false,false


In [44]:
# Preserve missingness evidence for every optional location attribute.
missingness(
    weekly_locations_raw,
    weekly_locations,
    [
        "shop_type",
        "postcode",
        "locality",
        "state",
        "geolocation_latitude",
        "geolocation_longitude",
    ],
)

column,raw_nulls,silver_nulls
str,i64,i64
"""shop_type""",436,436
"""postcode""",23,23
"""locality""",10,10
"""state""",10,10
"""geolocation_latitude""",6,6
"""geolocation_longitude""",6,6


In [45]:
# Inspect the eligible fallback population without silently resolving ambiguous keys.
locations.filter(pl.col("is_enrichment_safe")).select(
    "shop",
    "id",
    "country_code",
    "postcode",
    "geolocation_latitude",
    "geolocation_longitude",
).head()

shop,id,country_code,postcode,geolocation_latitude,geolocation_longitude
str,str,str,str,f64,f64
"""ldil""","""1a131""","""nl""",null,null,null
"""ldil""","""5f02f""","""de""",null,null,null
"""gc""","""f02af""","""be""","""9051""",51.015406,3.6528982
"""frc""","""64003""","""be""","""1780""",50.908945,4.3048594
"""frc""","""dd6cf""","""be""","""7010""",50.492171,3.9675296


In [46]:
# Report safe matches from the official enrichment result.
show_metrics(
    "weekly_locations", ["rows", "safe_fallback_matches", "context_mismatches"]
)

metric,value
str,str
"""rows""","""1230"""
"""safe_fallback_matches""","""0"""
"""context_mismatches""","""0"""


In [47]:
# Retain postcode and coordinate disagreement checks as visible monitoring evidence.
weekly_locations.select(
    [
        pl.col(c).sum().alias(c)
        for c in [
            "dq_postcode_conflict",
            "dq_latitude_conflict",
            "dq_longitude_conflict",
            "dq_enrichment_context_mismatch",
        ]
    ]
)

dq_postcode_conflict,dq_latitude_conflict,dq_longitude_conflict,dq_enrichment_context_mismatch
u32,u32,u32,u32
0,0,0,0


In [48]:
# Measure recovered geography and the gaps deliberately retained.
show_metrics("weekly_locations")

metric,value
str,str
"""rows""","""1230"""
"""safe_fallback_matches""","""0"""
"""context_mismatches""","""0"""
"""postcodes_enriched""","""0"""
"""coordinates_enriched""","""0"""
"""countries_enriched""","""0"""
"""remaining_missing_postcode""","""23"""
"""remaining_missing_coordinates""","""6"""


In [49]:
# Inspect the final reference without temporary fallback fields.
weekly_locations.head()

shop,location,location_name,shop_type,…,country_code,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,str,str,…,str,bool,bool,bool
"""idla""","""de""","""Default""",null,…,null,false,false,false
"""idla""","""binche""","""Binche""",null,…,null,false,false,false
"""idla""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,null,false,false,false
"""idla""","""lodelinsart""","""Lodelinsart""",null,…,null,false,false,false
"""idla""","""péruwelz""","""Péruwelz""",null,…,null,false,false,false


In [50]:
# Confirm the persisted weekly location checkpoint.
checkpoint_result("weekly_locations")

{'dataset': 'weekly_locations',
 'persisted_rows': 1230,
 'contracts': 'passed before publication'}

### 7.1 Conclusion

**Established:** 1,230 unique shop/location pairs; zero safe fallback matches and zero enriched fields.

**Caveats:** 23 missing postcodes, six missing coordinate pairs and 436 missing shop types remain.

**Silver decision:** Preserve gaps, shared coordinates and descriptive names. Country is populated only by safe evidence, never inferred from shop.


## 8. Weekly Prices

**Purpose / grain:** observations within `(daltix_id, shop, location, week)`; this working grain can contain conflicting price pairs.

**Issues → treatment:** collapse only exact six-field duplicates. Preserve different price pairs, mark ambiguity and derive source-specific promotion flags. The observation hash identifies a retained record; it does not repair the business grain or act as a Gold surrogate key.

**Contract:** complete fields, finite positive prices, no exact duplicates and consistent ambiguity flags. DuckDB writes directly from Parquet. [Implementation](../src/daltix_case/silver/weekly_prices.py).


In [51]:
# Inspect the source schema without materializing the 19.2M-row table in Python.
duck.sql("DESCRIBE raw_weekly_prices").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""shop""","""VARCHAR""","""YES""",null,null,null
"""location""","""VARCHAR""","""YES""",null,null,null
"""daltix_id""","""VARCHAR""","""YES""",null,null,null
"""week""","""DATE""","""YES""",null,null,null
"""price""","""DOUBLE""","""YES""",null,null,null
"""price_promo""","""DOUBLE""","""YES""",null,null,null


In [52]:
# Retain deduplication, ambiguity and promotion evidence from the official run.
display(show_metrics("weekly_prices"))
duck.sql("""
    SELECT count(*) FILTER (WHERE dq_source_exact_duplicate) AS retained_rows_from_exact_duplicates,
        count(*) FILTER (WHERE dq_repeated_grain) AS repeated_grain_rows,
        count(*) FILTER (WHERE is_price_resolved) AS non_ambiguous_rows,
        count(*) FILTER (WHERE dq_non_positive_price) AS non_positive_prices,
        count(*) FILTER (WHERE dq_non_positive_promo_price) AS non_positive_promos
    FROM weekly_prices
""").pl()

metric,value
str,str
"""raw_rows""","""19218071"""
"""rows""","""19075099"""
"""exact_duplicates_removed""","""142972"""
"""ambiguous_rows""","""751824"""
"""ambiguous_grains""","""375912"""
"""promotion_rows""","""1636220"""
"""promo_above_price_rows""","""10916"""


retained_rows_from_exact_duplicates,repeated_grain_rows,non_ambiguous_rows,non_positive_prices,non_positive_promos
i64,i64,i64,i64,i64
142972,751824,18323275,0,0


In [53]:
# Inspect examples of retained alternatives instead of selecting a supposed true weekly price.
duck.sql("""
    SELECT daltix_id, shop, location, week, price, price_promo, grain_observation_count,
        price_version_count, price_resolution_rule
    FROM weekly_prices WHERE dq_conflicting_price LIMIT 10
""").pl()

daltix_id,shop,location,week,…,price_promo,grain_observation_count,price_version_count,price_resolution_rule
str,str,str,date,…,f64,i64,i64,str
"""24419377ee9684c4296fff42be1e64…","""plc""","""ans""",2019-12-30,…,1.161,2,2,"""multiple_price_observations"""
"""24419377ee9684c4296fff42be1e64…","""plc""","""ans""",2019-12-30,…,2.19,2,2,"""multiple_price_observations"""
"""247661688edd2472c5acdd61e3c42f…","""frc""","""waterloo-centrum""",2020-12-28,…,9.42,2,2,"""multiple_price_observations"""
"""247661688edd2472c5acdd61e3c42f…","""frc""","""waterloo-centrum""",2020-12-28,…,14.49,2,2,"""multiple_price_observations"""
"""247e6d7a1e6cd82af949e60b18493e…","""frc""","""base""",2020-12-28,…,4.49,2,2,"""multiple_price_observations"""
"""247e6d7a1e6cd82af949e60b18493e…","""frc""","""base""",2020-12-28,…,4.939,2,2,"""multiple_price_observations"""
"""248b871e84488121715a191e559af4…","""frc""","""mont-st-jean""",2020-12-28,…,0.774,2,2,"""multiple_price_observations"""
"""248b871e84488121715a191e559af4…","""frc""","""mont-st-jean""",2020-12-28,…,0.946,2,2,"""multiple_price_observations"""
"""24974e57f3066cf69ef3e073a8a779…","""frc""","""drogenbos""",2019-12-30,…,2.09,2,2,"""multiple_price_observations"""


In [54]:
# Retain post-write evidence that exact duplicates and invalid resolution flags are absent.
duck.sql("""
    SELECT count(*) AS rows,
        count(*) - count(DISTINCT (daltix_id, shop, location, week, price, price_promo)) AS exact_duplicate_rows,
        count(*) FILTER (WHERE dq_conflicting_price AND is_price_resolved) AS invalid_resolution_rows
    FROM weekly_prices
""").pl()

rows,exact_duplicate_rows,invalid_resolution_rows
i64,i64,i64
19075099,0,0


### 8.1 Conclusion

**Established:** 19,218,071 Raw rows become 19,075,099 after removing 142,972 exact duplicates.

**Caveats:** 751,824 rows in 375,912 weekly grains remain ambiguous; 10,916 promotional values exceed regular price.

**Silver decision:** Retain alternatives and extremes. Gold must explicitly handle ambiguity before price aggregation or comparison; Silver does not select min/max/first as truth.


## 9. Cross-Table Contracts

**Purpose:** establish safe cardinality and context on the published Silver tables. Full weekly location coverage is required; incomplete product or nutritional overlap is reported, not hidden.

These are structural checks, not permission to treat later reference attributes as historically effective. [Official checks](../src/daltix_case/quality/cross_table.py).


In [55]:
# Confirm all seven outputs belong to this completed checkpoint.
pl.DataFrame([checkpoint_result(name) for name in OUTPUTS])

dataset,persisted_rows,contracts
str,i64,str
"""products""",32826,"""passed before publication"""
"""locations""",1638,"""passed before publication"""
"""prices""",1198547,"""passed before publication"""
"""nutritionals""",9467978,"""passed before publication"""
"""weekly_products""",114517,"""passed before publication"""
"""weekly_locations""",1230,"""passed before publication"""
"""weekly_prices""",19075099,"""passed before publication"""


In [56]:
# Retain weekly product coverage, row-preservation and context evidence.
show_metrics(
    "cross_table",
    [
        "weekly_price_rows",
        "weekly_product_joined_rows",
        "weekly_product_matched_rows",
        "weekly_product_unmatched_rows",
        "weekly_product_coverage_pct",
        "weekly_product_context_mismatches",
    ],
)

metric,value
str,str
"""weekly_price_rows""","""19075099"""
"""weekly_product_joined_rows""","""19075099"""
"""weekly_product_matched_rows""","""18323389"""
"""weekly_product_unmatched_rows""","""751710"""
"""weekly_product_coverage_pct""","""96.06"""
"""weekly_product_context_mismatc…","""0"""


In [57]:
# Retain the complete weekly location relationship and its cardinality evidence.
show_metrics(
    "cross_table",
    [
        "weekly_price_rows",
        "weekly_location_joined_rows",
        "weekly_location_unmatched_rows",
    ],
)

metric,value
str,str
"""weekly_price_rows""","""19075099"""
"""weekly_location_joined_rows""","""19075099"""
"""weekly_location_unmatched_rows""","""0"""


In [58]:
# Retain nutritional overlap and context evidence without joining temporal nutrient history to facts.
show_metrics(
    "cross_table",
    ["matched_nutritional_products", "nutrition_product_context_mismatches"],
)

metric,value
str,str
"""matched_nutritional_products""","""15696"""
"""nutrition_product_context_mism…","""0"""


### 9.1 Conclusion

**Established:** all table and cross-table contracts passed. Weekly product coverage is 96.06%; weekly location coverage is complete. Nutritionals overlap with 15,696 weekly products.

**Caveats:** 751,710 weekly price rows lack product metadata. Context agreement and safe cardinality do not establish temporal compatibility.

**Silver decision:** keep unmatched observations and separate sources. Define dimensional grain, unknown-member policy and historical enrichment rules explicitly in Gold.


## 10. Checkpoint and Next Phase

The complete pipeline now runs independently of this notebook. The persisted manifest records Raw hashes, source-code fingerprint, dependency versions, output schemas/hashes, metrics and timing.

**Next:** design Gold from these validated Silver tables. Address weekly ambiguity, incomplete reference coverage, nutritional portion/units and temporal compatibility before declaring analytical measures. No Gold transformation is implemented here.


In [59]:
# Show the final persisted artifact inventory rather than constructing an unpersisted manifest.
pl.DataFrame(
    [
        {
            "dataset": f["dataset"],
            "rows": f["rows"],
            "file": f["file"],
            "size_mb": round(f["size_bytes"] / 1024**2, 2),
        }
        for f in manifest["files"]
    ]
)

dataset,rows,file,size_mb
str,i64,str,f64
"""products""",32826,"""silver_products.parquet""",3.5
"""locations""",1638,"""silver_locations.parquet""",0.03
"""prices""",1198547,"""silver_prices.parquet""",3.92
"""nutritionals""",9467978,"""silver_nutritionals.parquet""",74.12
"""weekly_products""",114517,"""silver_weekly_products.parquet""",7.79
"""weekly_locations""",1230,"""silver_weekly_locations.parque…",0.05
"""weekly_prices""",19075099,"""silver_weekly_prices.parquet""",739.24


In [60]:
# Close the read-only analytical connection after all evidence has been displayed.
duck.close()
print("Silver checkpoint reviewed. Continue with explicit Gold modelling decisions.")

Silver checkpoint reviewed. Continue with explicit Gold modelling decisions.
